In [32]:
!pip install mediapipe opencv-python-headless


In [35]:

# Driver Drowsiness Detector - Google Colab (single cell)
# Uses MediaPipe Face Mesh + Eye Aspect Ratio (EAR)
from IPython.display import display, HTML

html_code = """
<div style="font-family: sans-serif;">
  <button id="startBtn" style="padding:8px 16px;font-size:16px;">▶ Start</button>
  <button id="stopBtn"  style="padding:8px 16px;font-size:16px;">■ Stop</button>
  <div id="status" style="font-size:24px;font-weight:bold;margin:10px 0;">Press Start</div>
  <video id="video" width="480" height="360" autoplay playsinline
         style="transform: scaleX(-1); border-radius:8px; background:#000;"></video>
</div>

<script>
// ---------- Settings (change these if needed) ----------
const EAR_THRESHOLD  = 0.22;  // eye is "closed" below this value
const CLOSED_SECONDS = 1.5;   // eyes closed this long = drowsy
const W = 480, H = 360;

// MediaPipe Face Mesh landmark points for each eye (6 points per eye)
const RIGHT_EYE = [33, 160, 158, 133, 153, 144];
const LEFT_EYE  = [362, 385, 387, 263, 373, 380];

// ---------- Helpers ----------
const statusDiv = document.getElementById('status');
const video     = document.getElementById('video');
let audioCtx = null, camera = null, faceMesh = null;
let closedSince = null, lastBeep = 0, running = false;

function loadScript(src) {
  return new Promise((resolve, reject) => {
    const s = document.createElement('script');
    s.src = src; s.onload = resolve;
    s.onerror = () => reject(new Error('Could not load ' + src));
    document.head.appendChild(s);
  });
}

function beep() {
  const osc = audioCtx.createOscillator();
  const gain = audioCtx.createGain();
  osc.type = 'square';
  osc.frequency.value = 900;
  gain.gain.value = 0.3;
  osc.connect(gain); gain.connect(audioCtx.destination);
  osc.start();
  osc.stop(audioCtx.currentTime + 0.3);
}

function dist(a, b) {            // distance in pixels (so width/height are not distorted)
  return Math.hypot((a.x - b.x) * W, (a.y - b.y) * H);
}

function eyeAspectRatio(lm, idx) {
  const [p1, p2, p3, p4, p5, p6] = idx.map(i => lm[i]);
  return (dist(p2, p6) + dist(p3, p5)) / (2 * dist(p1, p4));
}

// ---------- Runs on every camera frame ----------
function onResults(results) {
  if (!running) return;

  if (!results.multiFaceLandmarks || results.multiFaceLandmarks.length === 0) {
    closedSince = null;
    statusDiv.textContent = "No face detected";
    statusDiv.style.color = "orange";
    return;
  }

  const lm  = results.multiFaceLandmarks[0];
  const ear = (eyeAspectRatio(lm, LEFT_EYE) + eyeAspectRatio(lm, RIGHT_EYE)) / 2;
  const now = performance.now();

  if (ear < EAR_THRESHOLD) {
    if (closedSince === null) closedSince = now;
    const closedFor = (now - closedSince) / 1000;

    if (closedFor >= CLOSED_SECONDS) {               // DROWSY
      statusDiv.textContent = " DROWSY! WAKE UP! (EAR " + ear.toFixed(2) + ")";
      statusDiv.style.color = "red";
      if (now - lastBeep > 600) { beep(); lastBeep = now; }   // keeps beeping
    } else {                                         // just a blink
      statusDiv.textContent = "Eyes closed (EAR " + ear.toFixed(2) + ")";
      statusDiv.style.color = "orange";
    }
  } else {                                           // eyes open
    closedSince = null;
    statusDiv.textContent = "Alert (EAR " + ear.toFixed(2) + ")";
    statusDiv.style.color = "green";
  }
}

// ---------- Start / Stop ----------
document.getElementById('startBtn').onclick = async () => {
  if (running) return;
  try {
    statusDiv.textContent = "Loading face model...";
    statusDiv.style.color = "black";

    // Audio must be created inside a click, otherwise the browser blocks sound
    audioCtx = audioCtx || new (window.AudioContext || window.webkitAudioContext)();
    await audioCtx.resume();
    beep();                                          // test beep so you know sound works

    if (!faceMesh) {
      await loadScript('https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js');
      await loadScript('https://cdn.jsdelivr.net/npm/@mediapipe/face_mesh/face_mesh.js');
      faceMesh = new FaceMesh({
        locateFile: (f) => 'https://cdn.jsdelivr.net/npm/@mediapipe/face_mesh/' + f
      });
      faceMesh.setOptions({
        maxNumFaces: 1,
        refineLandmarks: true,
        minDetectionConfidence: 0.5,
        minTrackingConfidence: 0.5
      });
      faceMesh.onResults(onResults);
    }

    running = true;
    closedSince = null;
    camera = new Camera(video, {
      onFrame: async () => { await faceMesh.send({image: video}); },
      width: W, height: H
    });
    await camera.start();
    statusDiv.textContent = "Monitoring...";
  } catch (e) {
    running = false;
    statusDiv.textContent = " " + e.message;
    statusDiv.style.color = "red";
  }
};

document.getElementById('stopBtn').onclick = () => {
  running = false;
  if (camera) camera.stop();
  if (video.srcObject) video.srcObject.getTracks().forEach(t => t.stop());
  video.srcObject = null;
  statusDiv.textContent = "Stopped";
  statusDiv.style.color = "black";
};
</script>
"""

display(HTML(html_code))

This drowsiness detection model is successfully built for the **[SafePilot Project](https://github.com/maheshshinde9100/SafePilot)**. It is now fully operational and ready to be integrated into the main system!

https://github.com/maheshshinde9100/SafePilot